In [ ]:
from dtaidistance import dtw
import numpy as np
import librosa
ref_audio, sr = librosa.load("/home/alien/Git/DATA/ml-stuttering-events-dataset/CLIP_DIR/HeStutters/22/HeStutters_22_185.wav", sr=None)
target_audio, sr = librosa.load("/home/alien/Git/DATA/ml-stuttering-events-dataset/MeloTTS/HeStutters_22_185.wav", sr=sr) # Match sampling rate

dtw.distance(ref_audio, target_audio)

In [ ]:
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

ref_audio, sr = librosa.load("/home/alien/Git/DATA/ml-stuttering-events-dataset/CLIP_DIR/HeStutters/22/HeStutters_22_185.wav", sr=None)
target_audio, _ = librosa.load("/home/alien/Git/DATA/ml-stuttering-events-dataset/MeloTTS/HeStutters_22_185.wav", sr=sr)

ref_mfcc = librosa.feature.mfcc(y=ref_audio, sr=sr, n_mfcc=13)
target_mfcc = librosa.feature.mfcc(y=target_audio, sr=sr, n_mfcc=13)

D, wp = librosa.sequence.dtw(X=ref_mfcc, Y=target_mfcc, metric='cosine')

wp = np.array(wp[::-1])

hop_length = 512
ref_times = librosa.frames_to_samples(wp[:, 0], hop_length=hop_length)
target_times = librosa.frames_to_samples(wp[:, 1], hop_length=hop_length)

ref_times = np.clip(ref_times, 0, len(ref_audio) - 1)
target_times = np.clip(target_times, 0, len(target_audio) - 1)

aligned_audio = np.zeros_like(ref_audio)
for i in range(len(ref_times) - 1):
    ref_start = ref_times[i]
    ref_end = ref_times[i+1]
    target_start = target_times[i]
    target_end = target_times[i+1]

    segment = target_audio[target_start:target_end]
    duration = ref_end - ref_start

    if len(segment) > 0:
        segment_stretched = librosa.effects.time_stretch(segment.astype(np.float32), rate=len(segment)/duration)
        segment_stretched = segment_stretched[:duration]
        aligned_audio[ref_start:ref_start+len(segment_stretched)] = segment_stretched

sf.write("aligned_target_to_ref.wav", aligned_audio, sr)
print("Aligned audio saved as 'aligned_target_to_ref.wav'")


Aligned audio saved as 'aligned_target_to_ref.wav'


/home/alien/.pyenv/versions/3.12.11/envs/renv/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=512
  warnings.warn(
/tmp/ipykernel_2893859/2910075197.py:45: RuntimeWarning: divide by zero encountered in scalar divide
  segment_stretched = librosa.effects.time_stretch(segment.astype(np.float32), rate=len(segment)/duration)
